# Notebook used to test the models

### imports


In [34]:
%load_ext autoreload
%autoreload 2

from ml_powerlifting_potential.data import load_dataset
from sklearn.model_selection import train_test_split
from ml_powerlifting_potential.config import FEATURES, TARGETS, SEED
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,PolynomialFeatures

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
dataset = load_dataset()
dataset.describe()  # min, max, moyenne, quartiles de chaque colonne
dataset.isna().sum()  # doit afficher 0 partout

X = dataset[FEATURES]
y_squat = dataset[TARGETS[0]]
y_deadlift = dataset[TARGETS[1]]
X_train, X_test, y_squat_train, y_squat_test, y_deadlift_train, y_deadlift_test = train_test_split(X, y_squat,y_deadlift, test_size=0.2, random_state=SEED)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_squat_train shape: {y_squat_train.shape}")
print(f"y_squat_test shape: {y_squat_test.shape}")
print(f"y_deadlift_train shape: {y_deadlift_train.shape}")
print(f"y_deadlift_test shape: {y_deadlift_test.shape}")

X_train shape: (182100, 5)
X_test shape: (45525, 5)
y_squat_train shape: (182100,)
y_squat_test shape: (45525,)
y_deadlift_train shape: (182100,)
y_deadlift_test shape: (45525,)


In [30]:
bench_train = X_train["Best3BenchKg"]
bench_test = X_test["Best3BenchKg"]

# Gym rule: median of the per-athlete ratios, computed on the TRAIN set only
ratio_squat = (y_squat_train / bench_train).median()
ratio_deadlift = (y_deadlift_train / bench_train).median()
print(f"squat/bench = {ratio_squat:.2f}, deadlift/bench = {ratio_deadlift:.2f}")

# Predictions on the TEST set
pred_squat_rule = bench_test * ratio_squat
pred_deadlift_rule = bench_test * ratio_deadlift

def evaluate(y_true, y_pred) -> dict[str, float]:
    """MAE in kg (average error) and R2 (1 = perfect, 0 = no better than the mean)."""
    return {"MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred)}

print("Squat rule:", evaluate(y_squat_test, pred_squat_rule))
print("Deadlift rule:", evaluate(y_deadlift_test, pred_deadlift_rule))

squat/bench = 1.57, deadlift/bench = 1.85
Squat rule: {'MAE': 21.590190954021267, 'R2': 0.7480737990839819}
Deadlift rule: {'MAE': 28.404329357495882, 'R2': 0.5838059452247737}


In [32]:
linear_model_squat = LinearRegression()
linear_model_squat.fit(X_train, y_squat_train)

linear_model_deadlift = LinearRegression()
linear_model_deadlift.fit(X_train, y_deadlift_train)

pred_squat_linear = linear_model_squat.predict(X_test)
pred_deadlift_linear = linear_model_deadlift.predict(X_test)

print("Squat linear:", evaluate(y_squat_test, pred_squat_linear))
print("Deadlift linear:", evaluate(y_deadlift_test, pred_deadlift_linear))

Squat linear: {'MAE': 17.624369244378496, 'R2': 0.8310389597243819}
Deadlift linear: {'MAE': 19.318098649301696, 'R2': 0.8073293069867008}


In [37]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression


def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial features -> standardization -> linear regression."""
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("linreg", LinearRegression()),
    ])


targets = {
    "squat": (y_squat_train, y_squat_test),
    "deadlift": (y_deadlift_train, y_deadlift_test),
}

results = []
for degree in [2, 3]:
    for target_name, (y_train, y_test) in targets.items():
        model = make_poly_pipeline(degree)
        model.fit(X_train, y_train)
        for split_name, X_part, y_part in [("train", X_train, y_train), ("test", X_test, y_test)]:
            scores = evaluate(y_part, model.predict(X_part))
            results.append({"model": f"poly{degree}", "target": target_name, "split": split_name, **scores})

results_df = pd.DataFrame(results)
results_df

,model,target,split,MAE,R2
0,poly2,squat,train,17.329804,0.835871
1,poly2,squat,test,17.343826,0.836093
2,poly2,deadlift,train,18.673604,0.819935
3,poly2,deadlift,test,18.720340,0.818673
4,poly3,squat,train,17.279228,0.836732
5,poly3,squat,test,17.315425,0.836684
6,poly3,deadlift,train,18.578508,0.821771
7,poly3,deadlift,test,18.640255,0.820338
